# Clase 6: Entrenamiento — Pérdida, Gradient Descent y Backpropagation

## Ejemplos prácticos para la clase

---
## 1. Funciones de Pérdida (Loss Functions)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# MSE — Mean Squared Error (para regresión)

y_real = np.array([85, 72, 91, 60])      # notas reales
y_pred = np.array([82, 75, 88, 65])      # predicciones

errores = y_real - y_pred
mse = np.mean(errores ** 2)

print("=" * 50)
print("MSE — Mean Squared Error (Regresión)")
print("=" * 50)
print(f"\nReales:      {y_real}")
print(f"Predichos:   {y_pred}")
print(f"Errores:     {errores}")
print(f"Errores²:    {errores**2}")
print(f"\nMSE = promedio({errores**2}) = {mse}")
print(f"\nInterpretación: en promedio, el error al cuadrado es {mse}")
print(f"RMSE = √MSE = {np.sqrt(mse):.2f} (en las mismas unidades que las notas)")

In [ ]:
# Cross-Entropy: penaliza predicciones confiadas pero EQUIVOCADAS

# Visualizar por qué se usa logaritmo
p = np.linspace(0.01, 0.99, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cuando la respuesta correcta es 1
loss_1 = -np.log(p)
axes[0].plot(p, loss_1, color="#e74c3c", linewidth=2)
axes[0].set_xlabel("Probabilidad predicha P(y=1)")
axes[0].set_ylabel("Loss")
axes[0].set_title("Cuando la respuesta correcta es 1")
axes[0].annotate("Modelo dice 0.01\n(confiado pero EQUIVOCADO)\n→ Loss altísimo",
                 xy=(0.05, 4), fontsize=10, color="#e74c3c")
axes[0].annotate("Modelo dice 0.99\n(confiado y CORRECTO)\n→ Loss bajísimo",
                 xy=(0.55, 0.5), fontsize=10, color="#2ecc71")

# Cuando la respuesta correcta es 0
loss_0 = -np.log(1 - p)
axes[1].plot(p, loss_0, color="steelblue", linewidth=2)
axes[1].set_xlabel("Probabilidad predicha P(y=1)")
axes[1].set_ylabel("Loss")
axes[1].set_title("Cuando la respuesta correcta es 0")
axes[1].annotate("Modelo dice 0.99\n(confiado pero EQUIVOCADO)\n→ Loss altísimo",
                 xy=(0.6, 4), fontsize=10, color="#e74c3c")

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 5)

plt.suptitle("Binary Cross-Entropy: penaliza predicciones confiadas pero equivocadas", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-Entropy en PyTorch

# Binary Cross-Entropy
y_pred_t = torch.tensor([0.9, 0.1, 0.8, 0.3])
y_real_t = torch.tensor([1.0, 0.0, 1.0, 0.0])

bce = nn.BCELoss()
loss_bce = bce(y_pred_t, y_real_t)
print(f"Binary Cross-Entropy Loss: {loss_bce.item():.4f} (bajo, porque acertó bastante)")

# Categorical Cross-Entropy (multiclase)
criterion = nn.CrossEntropyLoss()
logits = torch.tensor([[2.0, 1.0, 0.1]])   # el modelo "piensa" que es clase 0
target = torch.tensor([0])                    # efectivamente es clase 0

loss_ce = criterion(logits, target)
print(f"\nCategorical Cross-Entropy Loss: {loss_ce.item():.4f} (bajo, porque acertó)")

# Cuando se equivoca:
target_mal = torch.tensor([2])                # la respuesta correcta era clase 2
loss_ce_mal = criterion(logits, target_mal)
print(f"Categorical CE (equivocado):    {loss_ce_mal.item():.4f} (alto, porque erró)")

print("\n" + "=" * 50)
print("¿Cuál usar?")
print("  Regresión (predecir número)      → MSE")
print("  Clasificación binaria (sí/no)    → Binary Cross-Entropy + Sigmoid")
print("  Clasificación multiclase (N)     → Cross-Entropy + Softmax")

---
## 2. Gradient Descent: bajando la montaña

In [ ]:
# Gradient Descent para encontrar el mínimo de f(w) = (w - 3)²
# El mínimo está en w = 3

w = 10.0          # empezamos lejos
lr = 0.1          # learning rate
historia = [(w, (w - 3) ** 2)]

print("Gradient Descent: minimizar f(w) = (w-3)²")
print(f"{'Paso':>5} {'w':>10} {'Loss':>12} {'Gradiente':>12}")
print("-" * 42)

for i in range(30):
    grad = 2 * (w - 3)            # derivada de (w-3)²
    w = w - lr * grad              # actualización
    loss = (w - 3) ** 2
    historia.append((w, loss))
    if i % 5 == 0:
        print(f"{i:>5d} {w:>10.4f} {loss:>12.6f} {grad:>12.4f}")

print(f"\nConvergió a w = {w:.6f} (óptimo: 3.0)")

In [ ]:
# Visualizar el descenso

w_range = np.linspace(-2, 12, 200)
loss_range = (w_range - 3) ** 2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Paisaje del loss con los pasos
axes[0].plot(w_range, loss_range, color="steelblue", linewidth=2)
ws = [h[0] for h in historia]
ls = [h[1] for h in historia]
for i in range(len(ws) - 1):
    alpha = 0.3 + 0.7 * (i / len(ws))
    axes[0].plot(ws[i], ls[i], 'o', color='#e74c3c', markersize=8, alpha=alpha)
    axes[0].annotate('', xy=(ws[i+1], ls[i+1]), xytext=(ws[i], ls[i]),
                     arrowprops=dict(arrowstyle='->', color='#e74c3c', alpha=alpha))
axes[0].set_xlabel('w')
axes[0].set_ylabel('Loss = (w-3)²')
axes[0].set_title('Gradient Descent: bajando la montaña')
axes[0].grid(True, alpha=0.3)

# Loss por paso
axes[1].plot(ls, 'o-', color='#e74c3c', markersize=4)
axes[1].set_xlabel('Paso')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss a lo largo del entrenamiento')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Learning Rate: el tamaño del paso

In [ ]:
# Efecto del learning rate

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, lr, titulo, color in [
    (axes[0], 0.01, "lr = 0.01 (muy lento)", "#2ecc71"),
    (axes[1], 0.1, "lr = 0.1 (justo)", "steelblue"),
    (axes[2], 0.95, "lr = 0.95 (muy alto, rebota)", "#e74c3c"),
]:
    w = 10.0
    hist = [(w, (w-3)**2)]
    for _ in range(50):
        grad = 2 * (w - 3)
        w = w - lr * grad
        hist.append((w, (w-3)**2))
    
    ws = [h[0] for h in hist]
    ls = [h[1] for h in hist]
    
    ax.plot(w_range, loss_range, color="gray", linewidth=1, alpha=0.5)
    for i in range(min(len(ws)-1, 30)):
        alpha = 0.3 + 0.7 * (i / 30)
        ax.plot(ws[i], ls[i], 'o', color=color, markersize=6, alpha=alpha)
    ax.set_xlabel('w')
    ax.set_ylabel('Loss')
    ax.set_title(titulo)
    ax.set_ylim(-5, 60)
    ax.grid(True, alpha=0.3)

plt.suptitle("Efecto del Learning Rate en Gradient Descent", fontsize=14)
plt.tight_layout()
plt.show()

print("Learning rate muy bajo → converge pero LENTO")
print("Learning rate justo → converge RÁPIDO")
print("Learning rate muy alto → REBOTA y no converge (puede divergir)")

---
## 4. Backpropagation: la cadena de la culpa

In [ ]:
# Backpropagation paso a paso con PyTorch autograd
# Red simple: 1 input → 1 oculta → 1 salida

# Datos
x = torch.tensor(2.0)
y = torch.tensor(1.0)  # valor real

# Pesos (con tracking de gradientes)
w1 = torch.tensor(0.5, requires_grad=True)
w2 = torch.tensor(-0.3, requires_grad=True)

print("=" * 60)
print("BACKPROPAGATION PASO A PASO")
print("=" * 60)

# Forward pass
z1 = x * w1
a1 = torch.sigmoid(z1)
z2 = a1 * w2
y_hat = torch.sigmoid(z2)
loss = (y - y_hat) ** 2

print(f"\n--- Forward Pass ---")
print(f"x = {x.item()}")
print(f"z1 = x * w1 = {x.item()} * {w1.item()} = {z1.item():.4f}")
print(f"a1 = sigmoid(z1) = {a1.item():.4f}")
print(f"z2 = a1 * w2 = {a1.item():.4f} * {w2.item()} = {z2.item():.4f}")
print(f"ŷ  = sigmoid(z2) = {y_hat.item():.4f}")
print(f"Loss = (y - ŷ)² = ({y.item()} - {y_hat.item():.4f})² = {loss.item():.4f}")

# Backward pass (PyTorch calcula TODOS los gradientes automáticamente)
loss.backward()

print(f"\n--- Backward Pass (Backpropagation) ---")
print(f"∂Loss/∂w1 = {w1.grad.item():.6f}")
print(f"∂Loss/∂w2 = {w2.grad.item():.6f}")

# Actualizar pesos
lr = 0.1
print(f"\n--- Actualización de pesos (lr={lr}) ---")
print(f"w1: {w1.item():.4f} → {w1.item() - lr * w1.grad.item():.4f}")
print(f"w2: {w2.item():.4f} → {w2.item() - lr * w2.grad.item():.4f}")

with torch.no_grad():
    w1 -= lr * w1.grad
    w2 -= lr * w2.grad

print(f"\nDespués de un paso, los pesos se ajustaron para reducir el error.")
print(f"Repetí esto miles de veces y el modelo aprende.")

---
## 5. Epochs, Batches e Iteraciones

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Simular un dataset de 1000 muestras
n_samples = 1000
X_demo = torch.randn(n_samples, 10)
y_demo = torch.randint(0, 2, (n_samples,))

batch_sizes = [32, 64, 128, 256]

print(f"Dataset: {n_samples} muestras")
print(f"\n{'Batch Size':>12} {'Iteraciones/Epoch':>20} {'Total pasos (5 epochs)':>25}")
print("-" * 60)
for bs in batch_sizes:
    iters = int(np.ceil(n_samples / bs))
    print(f"{bs:>12} {iters:>20} {iters * 5:>25}")

print(f"\n1 Epoch = 1 pasada completa por todo el dataset")
print(f"1 Batch = subconjunto del dataset procesado junto")
print(f"1 Iteración = 1 forward + backward + update")

In [ ]:
# Loop de entrenamiento completo con batches

# Modelo simple
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

dataset = TensorDataset(X_demo, y_demo)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20
history = []

for epoch in range(num_epochs):
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_X, batch_y in dataloader:
        # 1. Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # 2. Backward pass
        optimizer.zero_grad()  # limpiar gradientes anteriores
        loss.backward()        # calcular gradientes
        
        # 3. Actualizar pesos
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()
    
    acc = 100. * correct / total
    avg_loss = total_loss / len(dataloader)
    history.append({'loss': avg_loss, 'acc': acc})
    
    if epoch % 5 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch+1:>3d}/{num_epochs} - Loss: {avg_loss:.4f} - Accuracy: {acc:.1f}%")

In [ ]:
# Visualizar el entrenamiento

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, num_epochs + 1)
axes[0].plot(epochs_range, [h['loss'] for h in history], 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss durante el entrenamiento')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [h['acc'] for h in history], 'o-', color='#2ecc71', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy durante el entrenamiento')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Loop de entrenamiento: el loss baja y el accuracy sube', fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Tipos de Gradient Descent

In [ ]:
# Comparar: Batch GD vs SGD vs Mini-Batch GD

np.random.seed(42)

# Datos sintéticos
X_synth = np.random.randn(200, 1) * 2
y_synth = 3 * X_synth + 2 + np.random.randn(200, 1) * 0.5

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, batch_size, titulo, color in [
    (axes[0], 200, "Batch GD (todo el dataset)", "steelblue"),
    (axes[1], 1, "SGD (1 muestra)", "#e74c3c"),
    (axes[2], 32, "Mini-Batch GD (32 muestras)", "#2ecc71"),
]:
    w, b = 0.0, 0.0
    lr = 0.01
    losses = []
    
    for epoch in range(50):
        indices = np.random.permutation(200)
        epoch_loss = 0
        n_batches = 0
        
        for start in range(0, 200, batch_size):
            idx = indices[start:start+batch_size]
            X_batch = X_synth[idx]
            y_batch = y_synth[idx]
            
            y_pred = w * X_batch + b
            error = y_pred - y_batch
            
            grad_w = 2 * np.mean(X_batch * error)
            grad_b = 2 * np.mean(error)
            
            w -= lr * grad_w
            b -= lr * grad_b
            
            epoch_loss += np.mean(error ** 2)
            n_batches += 1
        
        losses.append(epoch_loss / n_batches)
    
    ax.plot(losses, color=color, linewidth=1.5, alpha=0.8)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f"{titulo}\nbatch_size={batch_size}")
    ax.grid(True, alpha=0.3)

plt.suptitle('Tipos de Gradient Descent', fontsize=14)
plt.tight_layout()
plt.show()

print("Batch GD:      Gradiente suave pero lento, necesita mucha memoria")
print("SGD:           Muy ruidoso pero rápido, puede escapar mínimos locales")
print("Mini-Batch GD: El balance perfecto (el estándar en la práctica)")

---
## 7. Optimizadores Modernos

In [ ]:
# Comparar SGD vs SGD+Momentum vs Adam

import torchvision
import torchvision.transforms as transforms

# Dataset MNIST para la comparación
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
loader = DataLoader(train_data, batch_size=128, shuffle=True)

def crear_modelo():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )

optimizadores = {
    "SGD (lr=0.01)": lambda m: torch.optim.SGD(m.parameters(), lr=0.01),
    "SGD + Momentum": lambda m: torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9),
    "Adam (lr=0.001)": lambda m: torch.optim.Adam(m.parameters(), lr=0.001),
}

resultados = {}
num_epochs = 5

for nombre, crear_opt in optimizadores.items():
    torch.manual_seed(42)
    model = crear_modelo()
    optimizer = crear_opt(model)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    for epoch in range(num_epochs):
        total_loss = 0
        for images, labels in loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader)
        losses.append(avg_loss)
    
    resultados[nombre] = losses
    print(f"{nombre:<20} - Loss final: {losses[-1]:.4f}")

# Gráfico
plt.figure(figsize=(10, 5))
for nombre, losses in resultados.items():
    plt.plot(range(1, num_epochs+1), losses, 'o-', linewidth=2, label=nombre)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Comparación de Optimizadores')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nAdam es el default recomendado para la mayoría de los casos.")

---
## Resumen

| Concepto | Clave |
|----------|-------|
| Loss Function | Mide qué tan equivocado está el modelo |
| MSE | Para regresión. Penaliza errores grandes |
| Cross-Entropy | Para clasificación. Penaliza confianza equivocada |
| Gradient Descent | `w_nuevo = w_viejo - lr × gradiente` |
| Learning Rate | Muy alto → diverge. Muy bajo → no aprende |
| Backpropagation | Calcula gradientes de atrás hacia adelante |
| Epoch | 1 pasada por todo el dataset |
| Batch | Subconjunto procesado junto (32-256) |
| Adam | Optimizador default. Rápido y estable |